In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')


In [ ]:
pd.set_option('display.max_rows', None)  # Show all rows
print(train.isnull().sum())
pd.reset_option('display.max_rows')      # Reset after

In [ ]:
# Fill categorical missing values with 'None'
cols_fill_none = [
    'Alley', 'FireplaceQu', 'PoolQC', 'Fence', 
    'MiscFeature', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'BsmtQual', 
    'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'MasVnrType'
]

for col in cols_fill_none:
    if col in train.columns:
        train[col] = train[col].fillna('None')
    if col in test.columns:
        test[col] = test[col].fillna('None')

# Fill numeric columns with 0
for col in ['MasVnrArea', 'GarageYrBlt']:
    if col in train.columns:
        train[col] = train[col].fillna(0)
    if col in test.columns:
        test[col] = test[col].fillna(0)


train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())

test['LotFrontage'] = test.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
test['LotFrontage'] = test['LotFrontage'].fillna(test['LotFrontage'].median())

train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])
test['Electrical'] = test['Electrical'].fillna(test['Electrical'].mode()[0])




In [ ]:
# Encode all object (categorical) columns
cat_cols = train.select_dtypes(include=['object']).columns

le = LabelEncoder()
for col in cat_cols:
    le.fit(list(train[col].astype(str).values) + list(test[col].astype(str).values))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))


In [ ]:
X = train.drop(['Id', 'SalePrice'], axis=1)
y = train['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_val)
print("R²:", round(r2_score(y_val, y_pred_xgb), 4))
print("MAE:", round(mean_absolute_error(y_val, y_pred_xgb), 2))


In [ ]:
# Ensure same engineered features exist in test data
X_test = test.copy()

# Add engineered features (must match training)
X_test['TotalSF'] = X_test['TotalBsmtSF'] + X_test['1stFlrSF'] + X_test['2ndFlrSF']
X_test['TotalBath'] = X_test['FullBath'] + X_test['HalfBath'] * 0.5

# Ensure test columns match training features
X_test = X_test[X_train.columns]

# Predict on test dataset
y_pred_test = xgb_model.predict(X_test)

# Create submission file
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': y_pred_test
})

submission.to_csv('submission.csv', index=False)
print("✅ submission.csv file created successfully!")
